In [1]:
import os
import json
import asyncio

In [2]:
from typing import Annotated, Sequence, TypedDict

# Set up tools for LLM

## Set up RAG tool
Reload the vectorstore. Firstly prepare the embedding model. Then reuse the vector store built in 04-RAG-Agentic-workflow

In [3]:
UK_DESTINATIONS = [ #A
    "Cornwall",
    "North_Cornwall",
    "South_Cornwall",
    "West_Cornwall",
]

persist_directory = "./chroma_langchain_db"

In [4]:
from langchain_google_vertexai import VertexAIEmbeddings

embedding = VertexAIEmbeddings(model="gemini-embedding-001")

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_13868\955259049.py:3: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding = VertexAIEmbeddings(model="gemini-embedding-001")
C:\Users\andrewchan\AppData\Local\Temp\ipykernel_13868\955259049.py:3: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embedding = VertexAIEmbeddings(model="gemini-embedding-001")


In [5]:
from langchain_community.vectorstores import Chroma

vectordb_client = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

vectordb_retriever = vectordb_client.as_retriever()

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_13868\1888515854.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb_client = Chroma(


In [8]:
from langchain_core.tools import tool

In [9]:
@tool(description="Search travel information about destinations in England.") #A
def search_travel_info(query: str) -> str: #B
    """Search embedded WikiVoyage content for information about destinations in England."""
    docs = vectordb_retriever.invoke(query) #C
    top = docs[:4] if isinstance(docs, list) else docs #C
    return "\n---\n".join(d.page_content for d in top) #D

## Set up weather info tool

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient

async def generate_accweather_tools():
    mcp_client = MultiServerMCPClient({
        "accuweather": {
            "url": "http://127.0.0.1:8020/accu-mcp-server",
            "transport": "streamable_http"
        }
    })
    return await mcp_client.get_tools()

# Build the agent

In [11]:
### Prepare the Agent State
import operator
from langgraph.managed.is_last_step import RemainingSteps
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

class AgentState(TypedDict): #A
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps

In [10]:
from langgraph.prebuilt import create_react_agent

async def build_agent(llm_model):
    accuweather_tools = await generate_accweather_tools()
    tools = [search_travel_info, *accuweather_tools]
    agent = create_react_agent(
        model=llm_model,
        tools=tools,
        state_schema=AgentState,
        prompt="""
        You are a helpful assistant that can 
        search travel information and get the weather forecast. 
        Only use the tools to find the information you need 
        (including town names).
        """
    )
    return agent

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # "gemini-2.5-flash-lite" for cheaper use, "gemini-2.5-flash" for normal use
    vertexai=True
)

In [13]:
travel_info_agent = await build_agent(llm)

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_13868\4113710464.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## Chat with agent

In [14]:
async def chat_loop(agent):
    print("UK Travel Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip() #C
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        result = await agent.ainvoke(state)
        response_msg = result["messages"][-1]
        print(f"Assistant: {response_msg.content}\n")

In [15]:
await chat_loop(travel_info_agent)

UK Travel Assistant (type 'exit' to quit)


You:  What's the weather in Penzance?


Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


Assistant: Partly sunny in Penzance with a temperature of 9 degrees Celsius and 62% relative humidity.



You:  Are there any hotel or BnB rooms available in Penzance?


Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


Assistant: I can only search for information about destinations in England. I cannot check for the availability of hotels or BnB rooms directly.



You:  exit
